In [37]:
import torch
from torch import nn

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

device

device(type='cuda')

In [38]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
).to(device)

inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]], device='cuda:0')

In [39]:
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out, device = torch.device("cuda"), qkv_bias = False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias = qkv_bias).to(device)
        self.W_key = nn.Linear(d_in, d_out, bias = qkv_bias).to(device)
        self.W_value = nn.Linear(d_in, d_out, bias = qkv_bias).to(device)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_score = queries @ keys.T
        attn_weights = torch.softmax(
            attn_score / (keys.shape[-1] ** 0.5), dim = -1
        )

        context_vector = attn_weights @ values

        return context_vector

In [40]:
d_in = inputs.shape[1]

d_out = 2

In [41]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v1(d_in, d_out)

queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_scores

tensor([[ 0.2899,  0.0716,  0.0760, -0.0138,  0.1344, -0.0511],
        [ 0.4656,  0.1723,  0.1751,  0.0259,  0.1771,  0.0085],
        [ 0.4594,  0.1703,  0.1731,  0.0259,  0.1745,  0.0090],
        [ 0.2642,  0.1024,  0.1036,  0.0186,  0.0973,  0.0122],
        [ 0.2183,  0.0874,  0.0882,  0.0177,  0.0786,  0.0144],
        [ 0.3408,  0.1270,  0.1290,  0.0198,  0.1290,  0.0078]],
       device='cuda:0', grad_fn=<MmBackward0>)

In [42]:
mask = torch.triu(torch.ones(attn_scores.shape[-1], attn_scores.shape[-1]), diagonal=1).to(device)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
masked

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]], device='cuda:0',
       grad_fn=<MaskedFillBackward0>)

In [43]:
# Make sure the sum of all values is one
attn_weights = torch.softmax(masked / (keys.shape[-1] ** 0.5), dim = 1)
attn_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]], device='cuda:0',
       grad_fn=<SoftmaxBackward0>)

In [44]:
context_vector = attn_weights @ sa_v2.W_value(inputs)
context_vector

tensor([[-0.0872,  0.0286],
        [-0.0991,  0.0501],
        [-0.0999,  0.0633],
        [-0.0983,  0.0489],
        [-0.0514,  0.1098],
        [-0.0754,  0.0693]], device='cuda:0', grad_fn=<MmBackward0>)

In [45]:
from torch import nn

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False, device = torch.device("cuda")):
        super().__init__()
        self.d_out = d_out
        self.dropout = nn.Dropout(dropout).to(device)

        self.W_query = nn.Linear(d_in, d_out, bias = qkv_bias).to(device)
        self.W_key = nn.Linear(d_in, d_out, bias = qkv_bias).to(device)
        self.W_value = nn.Linear(d_in, d_out, bias = qkv_bias).to(device)

        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length).to(device), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )

        attn_weights = torch.softmax(
            attn_scores / (keys.shape[-1] ** 0.5), dim = -1
        )

        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

In [46]:
torch.manual_seed(123)
batch = torch.stack((inputs, inputs), dim = 0)
batch.shape

torch.Size([2, 6, 3])

In [48]:
context_length = batch.shape[1]
ca = CausalAttention(d_in = 3, d_out = 2, context_length = 6, dropout = 0.0)
context_vec = ca(batch)
context_vec.shape

torch.Size([2, 6, 2])

In [49]:
context_vec

tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], device='cuda:0', grad_fn=<UnsafeViewBackward0>)